# Image patching

Load an RGB image and resize it to the experiment's resolution. Change the image path in the next cell to a photo on your machine. The patch extractor only rearranges pixels; normalization happens in the image-loading cell. Choose image dimensions divisible by the patch size.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.embedding import patchify, PatchEmbeddingModel
from utils.visualization import display_patches

torch.manual_seed(0)
np.random.seed(0)

image_size = 224
patch_size = 16
hidden_dim = 192

image = Image.open("data/image_8765cd2c.jpg").convert("RGB").resize((image_size, image_size))

image_tensor = torch.from_numpy(np.array(image)).permute(2, 0, 1).unsqueeze(0).float() / 255.0
results = patchify(image_tensor, patch_size=patch_size)
patches = results["patches"]

print(f"image tensor shape: {image_tensor.shape}")
print(f"patches tensor shape: {patches.shape}")
plt.imshow(image)
plt.axis("off")
plt.show()

# Patch projection, classification token, and learned positions

Create the model once and reuse it. It owns the linear projection, the shared classification token, and one learned position vector per token (including [CLS]). The returned patch embeddings are the raw projection; the complete token sequence also includes [CLS] and positions.

For 224 × 224 RGB images and 16 × 16 patches: [B, 3, 224, 224] → [B, 196, 768] → [B, 196, 192] → [B, 197, 192].

In [ ]:
patch_embedder = PatchEmbeddingModel(
    embedding_dim=results["embedding_dim"],
    hidden_dim=hidden_dim,
    num_patches=patches.size(1),
)
embedded_patches = patch_embedder(patches)

print(f"projected patches: {embedded_patches['embedded_patches'].shape}")
print(f"encoder input ([CLS] + patches + positions): {embedded_patches['patches_with_cls_token'].shape}")
print(f"shared classification parameter: {embedded_patches['cls_token'].shape}")
print(f"learned positions: {embedded_patches['positional_encoding'].shape}")

# Inspect gradient flow

This toy loss lets us inspect gradients in the embedding parameters before we build attention. It is only an autograd demonstration; the image-classification objective will come later.

In [ ]:
patch_embedder.zero_grad()
tokens = patch_embedder(patches)["patches_with_cls_token"]
loss = tokens.square().mean()
loss.backward()

for name, parameter in patch_embedder.named_parameters():
    print(f"{name}: shape={tuple(parameter.shape)}, gradient norm={parameter.grad.norm().item():.6f}")

# Transformer Encoder

Build one encoder block with token width 192, three attention heads, and an MLP width of 768 (four times the token width). Each head has width 64. These are small, tiny-style dimensions for this experiment.

The ViT block normalizes before each sublayer: input → LayerNorm → attention → residual addition → LayerNorm → MLP → residual addition. The residual paths bypass normalization. A complete encoder repeats this block and applies a final LayerNorm after the stack.

In [ ]:
from utils.transformer_encoder import TransformerEncoderLayer

transformer_encoder = TransformerEncoderLayer(hidden_dim=hidden_dim, num_heads=3, feedforward_dim=hidden_dim * 4, dropout=0.1)

## Follow gradients through attention

Call the two stages explicitly: raw flattened patches → embedding result dictionary → token tensor → encoder block. The loss below uses only the encoder's [CLS] output, so the patch projection can influence it through self-attention. This is still a toy autograd demonstration, not an image-classification loss. Tiny or zero gradients for individual parameters do not automatically mean the implementation is broken.

In [ ]:
patch_embedder.zero_grad()
transformer_encoder.zero_grad()

# The embedder returns a dictionary; the encoder consumes its token tensor.
embedding_outputs = patch_embedder(patches)
encoder_inputs = embedding_outputs["patches_with_cls_token"]
encoded_tokens = transformer_encoder(encoder_inputs)
cls_representation = encoded_tokens[:, 0, :]

print(f"encoder input: {encoder_inputs.shape}")
print(f"encoder output: {encoded_tokens.shape}")
print(f"classification-token representation: {cls_representation.shape}")

# Unlike the embedding-only check, this loss reads just the updated [CLS] token.
# Gradients reaching the patch projection now pass through attention.
loss = cls_representation.square().mean()
loss.backward()

for component_name, component in (
    ("patch_embedder", patch_embedder),
    ("transformer_encoder", transformer_encoder),
):
    for name, parameter in component.named_parameters():
        print(f"{component_name}.{name}: shape={tuple(parameter.shape)}, gradient norm={parameter.grad.norm().item():.6f}")